# Gaze-driven VLM input: Full gate (when) + foveated tokens (where), EgoEverything

One pipeline, both modules driven by the glasses' eye tracker and IMU:

```
gaze + IMU (10 Hz) -> Full gate (eye + head model) -> policy D: send a frame when the view
                                                      settles, and on each REFIXATION
sent frame (1344 px) + gaze point -> foveated tokens: 28 px cells at the gaze, 56 / 112 px
                                     cells shrunk to one token each further out
                                  -> Qwen2.5-VL-3B (its own vision encoder, no training)
```

[EgoEverything](https://huggingface.co/datasets/roxqtang/EgoEverything) (arXiv 2604.08342):
5-option questions about what an Aria wearer saw, built from real gaze. 834 questions fall
on 12 of the gate's videos (4 test videos; 8 training videos scored by the fold model that
held each out).

**The question this notebook answers:** with the frames the gate sends, does foveated
tokenization at the gaze beat uniform resizing **at the same number of tokens**, and is it
the gaze that matters (fovea at the image centre / at another moment's gaze as controls)?

| condition (all on policy D's frames) | tokens per frame |
|---|---|
| **fovea at gaze** (a = 40) | ~250 |
| uniform, same tokens as the fovea (per frame) | same |
| uniform 448 | 256 (the resolution used before) |
| fovea at image centre | ~250 |

**All 12 videos (default, 834 questions):** `DEMO = False` in section 3; about 45 min on an A100
(the 4 test videos' scores from the demo are reused). `DEMO = True` = the 4 test videos only.

## Run order

1. Clone, mount, install  2. AEA links  3. Questions  4. Full gate + policy D
5. Frames (1344 px, cached on Drive)  6. Gaze and spatial screen (no GPU)
7. Qwen: unit tests, parity with Qwen's own pipeline  8. Score  9. Results
10. Token saving: foveation on the 448 px frame vs uniform downscaling

Earlier versions of this notebook (D vs fixed-interval and content-based frame selection)
are in git history, commit `d45e982`.

## 1. Clone, mount, install

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/qwen_fovea.py'), f'vlmeval/qwen_fovea.py is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest'], check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video and size, reused later
LOCAL_HI = '/content/frames_1344'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the Full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('Full model checkpoints: 5 folds found in', CKPT_FULL)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

# DEMO = True: only the 4 locked TEST videos (204 questions), to check that everything works.
# DEMO = False: all 12 gate videos with EgoEverything questions (834 questions).
DEMO = False
qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
if DEMO:
    qs = qs[qs.video.isin(test_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Full gate + policy D: which frames are sent

Each training video is scored by the fold model that held it out; test videos by the 5-fold
ensemble (cached on Drive). The held-out 4-class accuracy should be close to the
cross-validation numbers (~0.6), not higher, which would mean a video leaked into its own
training. Policy D then sends a frame when the view settles after motion, and on each
REFIXATION.

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"held-out 4-class accuracy on these {len(videos)} videos: {(o12.pred == o12.quad).mean():.3f}")

labels = {'full': egoev.video_labels(clips, oof)}
sel = {'D: full': egoev.select_conditions(clips, labels, main='full')['D: full']}
D = sel['D: full']
per = pd.DataFrame({'video_s': n_frames, 'frames sent': {v: len(D[v]) for v in videos}})
per['one frame every (s)'] = (per.video_s / per['frames sent']).round(1)
display(per)
print(f"policy D sends {60 * per['frames sent'].sum() / per.video_s.sum():.1f} frames/min "
      f"({per['frames sent'].mean():.1f} per video)")

## 5. Frames at 1344 px (cached on Drive)

Every condition starts from the same 1344 px frame (AEA's 1408 px resized to 12 x 112, the
size the foveated layout needs); the uniform conditions resize it down. Frames are chosen
with the gate's own 1 fps rule, so frame k is frame k of the labels. First run downloads
each video once (~165 MB, needs fresh links), later runs read the tar from Drive.

**Alignment check:** the frame nearest a few keyframes with EgoEverything's box; the boxes
should sit on the named objects.

In [ ]:
from vlmeval.frames import ensure_frames, frame_path
from vlmeval import qwen_fovea as qf, spatial as sp

metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_HI, FRAMES_TAR, size=qf.IMG)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames at {qf.IMG} px  {flag}")

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

show = qs[qs.bbox.notna() & qs.t0.notna()].sample(min(6, len(qs)), random_state=1)
fig, ax = plt.subplots(1, len(show), figsize=(3.4 * len(show), 3.8))
for a_, q in zip(np.atleast_1d(ax), show.itertuples()):
    m = metas[q.video]
    k = int(np.argmin(np.abs(np.array(m['ts']) - q.t0)))
    img = plt.imread(frame_path(m, k))
    s = img.shape[1] / m['orig_w']
    x1, y1, x2, y2 = [b * s for b in q.bbox]
    a_.imshow(img)
    a_.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='yellow', lw=2))
    a_.set_title(f"{q.object}\nt0={q.t0:.1f}s  frame {k}", fontsize=8); a_.axis('off')
plt.tight_layout(); plt.show()

cov = egoev.coverage(qs, sel, frame_time)
print('policy D: how close its nearest sent frame is to each question\'s moment')
display((cov[['hit_1s', 'hit_2s', 'hit_5s']].mean() * 100).round(1).to_frame('% of questions').T
        .assign(median_dist_s=cov.dist.median()))

## 6. Gaze and spatial screen (no GPU)

Gaze comes from the fold CSVs' `gaze_xy_window`: MPS gaze already projected into the upright
frame with the factory calibration (no VRS download), ~10 samples per second. **One gaze point
per frame** = the densest fixation cluster among the samples of the second before the frame
(samples within 3 deg of each other, recent ones weighted more, exp(-dt / 0.5 s)); its
weighted centroid is the fovea. A plain mean could land between two fixations after a saccade.

This section checks, before the VLM:

- **A. On each question's keyframe:** is the target inside the 14 px-equivalent fovea, for
  the fovea at the gaze vs at the image centre vs at the same wearer's gaze from another
  moment? `ratio` = tokens the foveated layout spends on the target / tokens a uniform
  resize to the same total would spend (> 1 = more detail on the target at equal cost).
- **B. With policy D:** how often D's nearest sent frame is within 1 s of the keyframe AND
  has the target in its fovea.
- **C. Is the gaze a fixation when D sends?** Gaze spread within +-0.2 s of each frame D
  sends vs every 1 fps frame (the base rate): D sends at settles, so its frames should be
  fixations more often, which makes the fovea land where the eyes rest.

Caveats: EgoEverything questions are built from objects the wearer looked at, so gaze near
the target is partly by construction (the centre and other-time rows show how much). The box
is valid at the keyframe only. Eccentricity uses a pinhole camera; Aria RGB is fisheye.

In [ ]:
A_VALUES = (10, 20, 40, 80, 160)
A_FOV = 40                           # fall-off used for scoring below
STABLE_DEG = 3.0

tracks, n_drop = sp.gaze_tracks(FOLDS, videos, frame_time)
print(f'gaze tracks: {len(tracks)} videos, {sum(len(t) for t, _ in tracks.values()):,} samples '
      f'({n_drop} windows dropped for out-of-FOV fillers)')

def video_ci(d):
    lo, hi = egoev.video_bootstrap(d, lambda t: t['diff'].mean())
    return d['diff'].mean(), lo, hi

# A. keyframe
kf = sp.keyframe_screen(qs, tracks, metas, A_VALUES)
g = kf[(kf.source == 'gaze') & (kf.a == A_FOV)]
chk = g[['dist2gaze', 'gaze_box_centre_px']].apply(pd.to_numeric, errors='coerce').dropna()
print(f"projection check: Spearman(our gaze-to-box distance, dataset AveDist2Gaze) = "
      f"{chk.corr(method='spearman').iloc[0, 1]:.2f} on {len(chk)} questions (should be clearly > 0)")
kt = (kf[kf.a == A_FOV].groupby('source', sort=False)
        .agg(questions=('qid', 'nunique'), gaze_inside_box=('inside', 'mean'),
             median_deg_to_box=('d_deg', 'median'), target_in_fovea=('in_fovea', 'mean'),
             median_ratio=('ratio', 'median')))
kt[['gaze_inside_box', 'target_in_fovea']] *= 100
print(f'\nA. on the keyframe, a = {A_FOV} (% of questions):')
display(kt.round(2))
w = kf[kf.a == A_FOV].pivot_table(index=['qid', 'video'], columns='source',
                                  values=['in_fovea', 'ratio']).dropna()
for ctrl in ('centre', 'gaze, other time'):
    for col, scale, unit in (('in_fovea', 100, 'points'), ('ratio', 1, '')):
        d = pd.DataFrame({'video': w.index.get_level_values('video'),
                          'diff': scale * (w[(col, 'gaze')] - w[(col, ctrl)]).to_numpy()})
        m_, lo, hi = video_ci(d)
        print(f'   gaze - {ctrl:<17} {col:<9} {m_:+6.2f} {unit:<6} 95% CI [{lo:+.2f}, {hi:+.2f}]')

# B. with policy D
cb = sp.combined_screen(qs, sel, ['D: full'], tracks, metas, frame_time, a=A_FOV, tol=1.0)
gz, ce = cb[cb.source == 'gaze'], cb[cb.source == 'centre']
print(f"\nB. policy D: nearest sent frame within 1 s of the keyframe for {100 * gz.hit_time.mean():.1f}% of "
      f"questions; with the target in its fovea: gaze {100 * gz.hit_fovea.mean():.1f}%, "
      f"centre {100 * ce.hit_fovea.mean():.1f}%")

# C. fixation at D's frames vs every 1 fps frame
allf = {v: list(range(n_frames[v])) for v in videos}
st = sp.sent_frame_stability({'D: full': D, '1 fps: every frame': allf}, ['D: full', '1 fps: every frame'],
                             tracks, frame_time)
ct = st.groupby('condition', sort=False).agg(
    frames=('spread', 'size'), median_spread_deg=('spread', 'median'),
    fixating=('spread', lambda s_: 100 * (s_.dropna() < STABLE_DEG).mean()))
print(f'\nC. gaze spread within +-0.2 s of each frame (fixating = spread < {STABLE_DEG:g} deg):')
display(ct.round(2))
d = st.dropna(subset=['spread']).assign(fix=lambda t: (t.spread < STABLE_DEG).astype(float))
stat = lambda t: 100 * (t[t.condition == 'D: full'].fix.mean() - t[t.condition == '1 fps: every frame'].fix.mean())
lo, hi = egoev.video_bootstrap(d, stat)
print(f'   D - every frame: {stat(d):+.1f} points fixating, 95% CI [{lo:+.1f}, {hi:+.1f}]')

## 7. Qwen2.5-VL-3B: unit tests and parity with Qwen's own pipeline

Foveated tokens are built **before Qwen's vision encoder** (`vlmeval/qwen_fovea.py`): one Qwen
token = one 28 px cell (2 x 2 patches of 14 px); 56 / 112 px cells are shrunk to 28 px; each
cell's 4 patches get 2D RoPE at the centres of the cell's quadrants (Qwen's own integers for
28 px cells) and join the 112 px attention window holding the cell; the LLM gets one token
per cell with M-RoPE at the cell centre. Nothing is trained.

| check | pass |
|---|---|
| unit tests, tiny random Qwen2.5-VL encoder on CPU: layout covers every cell once, 28 px gaze cell, token count = section 6's, patches = Qwen's image processor, windows = Qwen's, **all-28 px layout through the overrides = stock encoder**, tokens follow their cells, M-RoPE rule, condition layouts | all pass |
| real model, one frame, all-28 px layout at 448 and 1344 px vs Qwen's own processor + model: pixels, vision features, token ids, M-RoPE positions, answer probabilities | identical up to bf16 noise, same answer |
| feature drift: cosine of foveated tokens with the full-resolution tokens they cover, by cell size | 28 px cells close to 1 |

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_qwen_fovea.py', 'tests/test_gaze_fixation.py', '-q', '-rs'],
                   capture_output=True, text=True, cwd=REPO)
import transformers, inspect
print('transformers', transformers.__version__, '| torch', torch.__version__)
print(r.stdout[-4000:], r.stderr[-2000:])
if r.returncode != 0:            # show what this transformers version's encoder looks like
    from transformers.models.qwen2_5_vl import modeling_qwen2_5_vl as mq
    for name in ('Qwen2_5_VisionTransformerPretrainedModel', 'Qwen2_5_VLVisionBlock'):
        obj = getattr(getattr(mq, name, None), 'forward', None)
        try:
            print(f'---- {name}.forward ----\n' + inspect.getsource(inspect.unwrap(obj))[:5000])
        except Exception as ex:
            print(f'---- {name}.forward: source not available ({ex})')
assert r.returncode == 0, 'unit tests failed: fix before using the GPU'

from vlmeval import qwen_mcq
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
model, proc, letter_ids, dtype = qwen_mcq.load_model(MODEL_ID)
print(MODEL_ID, dtype, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU')
ip = proc.image_processor

q0 = qs.iloc[0]
m0 = metas[q0.video]
k0 = int(np.argmin(np.abs(np.array(m0['ts']) - q0.t0)))
img0 = qf.load_frame01(frame_path(m0, k0))
for size in (448, 1344):
    rep = qf.parity_report(model, proc, letter_ids, img0, q0.question, q0.options,
                           qwen_mcq.build_messages, size=size)
    print(f"parity, all-28 px layout at {size} px vs Qwen's own pipeline:")
    for k_, v_ in rep.items():
        print(f'   {k_:<24} {v_}')
    ok = (rep['pixel_max_abs_diff'] < 1e-2 and rep['vision_cos_min'] > 0.999 and rep['tokens_ids_equal']
          and rep['probs_max_abs_diff'] < 0.02 and rep['same_answer']
          and (not isinstance(rep.get('positions_max_abs_diff'), float) or rep['positions_max_abs_diff'] == 0))
    assert ok, f'parity failed at {size} px: do not trust the foveated results'
print('parity OK\n')

g0 = sp.gaze_fixation(tracks[q0.video], m0['ts'][k0]) or (0.5, 0.5)
cells0 = qf.foveated_cells(*g0, a=A_FOV)
print(f'feature drift on one frame (a = {A_FOV}, {len(cells0)} tokens):')
display(qf.fovea_feature_drift(qf.visual_of(model), img0, cells0, ip.image_mean, ip.image_std).round(3))

COLS = {28: 'white', 56: 'yellow', 112: 'deepskyblue'}
ex = [(v, f) for v in videos[:3] for f in D[v][len(D[v]) // 2: len(D[v]) // 2 + 1]]
fig, axs = plt.subplots(1, len(ex), figsize=(4.4 * len(ex), 4.6))
for a_, (v, f) in zip(np.atleast_1d(axs), ex):
    gz_ = sp.gaze_fixation(tracks[v], frame_time(v, f)) or (0.5, 0.5)
    cells = qf.foveated_cells(*gz_, a=A_FOV)
    a_.imshow(plt.imread(frame_path(metas[v], f)))
    for y, x, s in cells:
        a_.add_patch(mpatches.Rectangle((x, y), s, s, fill=False, ec=COLS[s], lw=0.5))
    a_.plot(gz_[0] * qf.IMG, gz_[1] * qf.IMG, 'r+', ms=14, mew=2)
    a_.set_title(f"{v}, frame {f} (sent by D)\n{len(cells)} tokens: "
                 + ' / '.join(f"{(cells[:, 2] == s).sum()} x {s}px" for s in COLS), fontsize=8)
    a_.axis('off')
plt.suptitle(f'foveated cells at a = {A_FOV} (white 28 px, yellow 56 px, blue 112 px; + = gaze)', fontsize=9)
plt.tight_layout(); plt.show()

budget = pd.DataFrame({a: {'ring1_deg': sp.ring_radii_deg(sp.FOV / 2, a)[0],
                           'ring2_deg': sp.ring_radii_deg(sp.FOV / 2, a)[1],
                           'tokens per frame, D frames (mean)': np.mean(
                               [len(qf.foveated_cells(*(sp.gaze_fixation(tracks[v], frame_time(v, f)) or (0.5, 0.5)), a=a))
                                for v in videos for f in D[v]])}
                       for a in A_VALUES}).T.rename_axis('a')
print(f'token budget at {qf.IMG} px (uniform 448 = 256 tokens, uniform 1344 = 2304):')
display(budget.round(1))

## 8. Score: policy D's frames under four tokenizations

Every question gets all the frames policy D sent on its video (each labelled with its time),
tokenized per condition. Each frame's vision features are computed once per video and
condition and reused by the video's questions. Answer = argmax of the raw next-token logits
over A-E (the same for every condition). Resumable: a disconnected session continues where
it stopped.

`DEMO` in section 3 sets the videos: all 12 (834 questions) by default; the demo's scores are reused.

In [ ]:
RUN_VIDEOS = videos                      # set by DEMO in section 3

CONDS = {
    f'D + fovea at gaze (a={A_FOV})':          ('fovea', dict(a=A_FOV, gaze='gaze')),
    'D + uniform, same tokens as the fovea':   ('uniform_match', f'D + fovea at gaze (a={A_FOV})'),
    'D + uniform 448 (256 tokens)':            ('uniform', 448),
    f'D + fovea at image centre (a={A_FOV})':  ('fovea', dict(a=A_FOV, gaze='centre')),
}

frames_of = {}
for v in RUN_VIDEOS:
    m = metas[v]
    frames_of[v] = []
    for f in D[v]:
        if f >= m['n']:
            continue
        t = float(m['ts'][f])
        gz_ = sp.gaze_fixation(tracks[v], t) if v in tracks else None
        frames_of[v].append(dict(key=(v, int(f)), path=frame_path(m, f), t=t,
                                 gaze=tuple(map(float, gz_)) if gz_ is not None else None, gaze_other=None))
items = [dict(qid=int(q.qid), video=q.video, question=q.question, options=list(q.options),
              answer=q.answer, frames=frames_of[q.video])
         for q in qs[qs.video.isin(RUN_VIDEOS)].sort_values(['video', 'qid']).itertuples()]
nf = np.array([len(it['frames']) for it in items])
print(f"{len(items)} questions on {len(RUN_VIDEOS)} videos x {len(CONDS)} conditions; "
      f"frames per question: mean {nf.mean():.1f}, max {nf.max()}; "
      f"gaze missing on {sum(fr['gaze'] is None for v in RUN_VIDEOS for fr in frames_of[v])} of "
      f"{sum(len(frames_of[v]) for v in RUN_VIDEOS)} sent frames (fovea falls back to the centre there)")

OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_D_fovea.jsonl")
qf.run_questions(model, proc, letter_ids, items, CONDS, OUT, qwen_mcq.build_messages)

## 9. Results

- **Accuracy per condition** with a 95% interval from resampling the videos, and the image
  tokens each condition used per question.
- **Paired comparisons** on the same questions (`diff` = a - b in accuracy points;
  `a_only_right` / `b_only_right` = questions only one condition got right; `p_mcnemar`).
  The main one: **fovea at gaze vs uniform at the same tokens.**
- **By whether the target was in the gaze fovea** on the keyframe (section 6A).
- **Per video**, so one video with many questions cannot drive the result unseen.

In [ ]:
res = egoev.load_results(OUT)
res = res[res.video.isin(RUN_VIDEOS) & res.condition.isin(CONDS)]
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} questions not scored, left out:'); display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy()
res['correct'] = res['correct'].astype(int)
res['split'] = np.where(res.video.isin(test_videos), 'test', 'train (held out)')

acc = egoev.accuracy_table(res)
acc['image tokens / question'] = res.groupby('condition').n_image_tokens.mean()
acc['tokens / frame'] = acc['image tokens / question'] / acc['frames_per_q']
acc['frames with no gaze'] = res.groupby('condition').n_gaze_missing.mean()
acc = acc.loc[[c for c in CONDS if c in acc.index]]
print(f'{res.qid.nunique()} questions on {res.video.nunique()} videos, policy D frames:')
display(acc.drop(columns=['frames with no gaze']).round(2))

G, M, U, C = list(CONDS)
pairs = [(G, M), (G, U), (G, C), (M, U)]
pt = pd.DataFrame([egoev.paired_test(res, a, b) for a, b in pairs if {a, b} <= set(res.condition)])
print('paired on the same questions (diff = a - b, accuracy points):')
display(pt.round(3))

inf = kf[(kf.source == 'gaze') & (kf.a == A_FOV)].set_index('qid').in_fovea.astype(bool)
sub = res[res.condition.isin([G, M])].assign(target_in_gaze_fovea=lambda d: d.qid.map(inf)).dropna(
    subset=['target_in_gaze_fovea'])
print('accuracy (%) by whether the target was inside the gaze fovea on the keyframe:')
display((100 * sub.pivot_table(index='target_in_gaze_fovea', columns='condition', values='correct'))
        .round(1).join(sub[sub.condition == G].groupby('target_in_gaze_fovea').size().rename('questions')))

print('by split:')
display((100 * res.pivot_table(index='split', columns='condition', values='correct')).round(1))
per_v = 100 * res.pivot_table(index='video', columns='condition', values='correct')
per_v['questions'] = res[res.condition == G].groupby('video').size()
print('per video (%):'); display(per_v.round(1))

## 10. Token saving: foveation on the frame Qwen normally sees (448 px)

Section 9 matched the foveated budget to uniform 448 (~256 tokens) to compare accuracy at
equal cost. Here, as in the LLaVA experiment, the **same 448 px frame** is foveated to
**fewer** tokens: 28 px cells (full resolution) at the gaze, 56 / 112 px cells shrunk to one
token further out. Budgets of about 48 / 96 / 150 / 205 tokens per frame (vs 256) come from
the rings' fall-off `a` and maximum eccentricity `R` (the spatial module's two FOVI
parameters), **fixed from the 4-test-video demo** so the 12-video run is not tuned on itself.

| condition (all on policy D's frames) | role |
|---|---|
| uniform 448 (256 tokens) | Qwen's normal input (scored in section 8, reused) |
| fovea@448 at gaze, ~B tokens | **the saving**: how much accuracy is kept at fewer tokens |
| uniform, same tokens as fovea@448 ~B | **the honest baseline**: just downscale the frame to the same token count |
| fovea@448 at image centre, middle budget | is it the gaze? |

A token saving counts only if the fovea curve sits **above** the uniform-downscaling curve at
the same tokens; matching uniform 448 alone is not enough, since plain downscaling also saves
tokens. About 25 minutes on an A100 for all 12 videos (the 4 demo videos are reused).

In [ ]:
# 10a. pick (R, a) for each token budget on the 448 px frame, from the gaze at D's frames
LOW_IMG = 448
# Ring settings (max eccentricity R, fall-off a) FIXED from the 4-test-video demo, where they were
# picked to hit ~48 / 96 / 150 / 205 tokens per frame. Fixing them means the 12-video run is not
# tuned on its own videos, and the demo's scores (same condition names) are reused.
PICKS = [dict(R=70, a=160, label=48), dict(R=180, a=40, label=96),
         dict(R=180, a=160, label=150), dict(R=240, a=160, label=205)]

gz_D = [fr['gaze'] or (0.5, 0.5) for v in RUN_VIDEOS for fr in frames_of[v]]
picks = {p['label']: dict(p, tokens=np.mean([len(qf.foveated_cells(*g_, a=p['a'], img=LOW_IMG, R_deg=p['R']))
                                             for g_ in gz_D])) for p in PICKS}
pk = pd.DataFrame(picks).T.rename_axis('setting (demo tokens)')
pk['ring1_deg'] = [sp.ring_radii_deg(R, a)[0] for R, a in zip(pk.R, pk.a)]
pk['ring2_deg'] = [sp.ring_radii_deg(R, a)[1] for R, a in zip(pk.R, pk.a)]
pk['saving vs 256'] = (100 * (1 - pk.tokens / 256)).round(1).astype(str) + '%'
print(f'ring settings on the {LOW_IMG} px frame; tokens = mean over the {len(gz_D)} frames D sends on these videos:')
display(pk.round(1))

def fov448(p, src='gaze'):
    return f"D + fovea@448 {src}, ~{p['label']} tok (R={p['R']:g}, a={p['a']:g})"

U448 = 'D + uniform 448 (256 tokens)'
SAVE_CONDS = {U448: ('uniform', 448)}
for p in picks.values():
    SAVE_CONDS[fov448(p)] = ('fovea', dict(a=p['a'], R=p['R'], img=LOW_IMG, gaze='gaze'))
    SAVE_CONDS[f"D + uniform, same tokens as fovea@448 ~{p['label']}"] = ('uniform_match', fov448(p))
p_mid = list(picks.values())[len(picks) // 2]
SAVE_CONDS[fov448(p_mid, 'centre')] = ('fovea', dict(a=p_mid['a'], R=p_mid['R'], img=LOW_IMG, gaze='centre'))
print('conditions:', *SAVE_CONDS, sep='\n   ')

# what the model gets at each budget, on one frame sent by D
v_, fr_ = RUN_VIDEOS[0], frames_of[RUN_VIDEOS[0]][len(frames_of[RUN_VIDEOS[0]]) // 2]
img_ = qf.resize01(qf.load_frame01(fr_['path']), LOW_IMG).permute(1, 2, 0).numpy()
g_ = fr_['gaze'] or (0.5, 0.5)
fig, axs = plt.subplots(1, len(picks) + 1, figsize=(3.6 * (len(picks) + 1), 3.9))
axs[0].imshow(img_); axs[0].set_title('uniform 448: 256 tokens', fontsize=8); axs[0].axis('off')
for a_, p in zip(axs[1:], picks.values()):
    cells = qf.foveated_cells(*g_, a=p['a'], img=LOW_IMG, R_deg=p['R'])
    a_.imshow(img_)
    for y, x, s in cells:
        a_.add_patch(mpatches.Rectangle((x, y), s, s, fill=False, ec=COLS[s], lw=0.6))
    a_.plot(g_[0] * LOW_IMG, g_[1] * LOW_IMG, 'r+', ms=12, mew=2)
    a_.set_title(f"{len(cells)} tokens (R={p['R']:g}, a={p['a']:g})", fontsize=8); a_.axis('off')
plt.suptitle(f'{v_}, frame {fr_["key"][1]}: white 28 px (full res), yellow 56 px, blue 112 px; + = gaze', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 10b. score (same resumable file as section 8, so uniform 448 is reused)
qf.run_questions(model, proc, letter_ids, items, SAVE_CONDS, OUT, qwen_mcq.build_messages)

In [ ]:
# 10c. accuracy vs tokens: does foveation keep more accuracy than plain downscaling?
rs = egoev.load_results(OUT)
REF = [c for c in ('D + fovea at gaze (a=40)', 'D + uniform, same tokens as the fovea') if c in set(rs.condition)]
rs = rs[rs.video.isin(RUN_VIDEOS) & rs.condition.isin(list(SAVE_CONDS) + REF)]
rs = rs[rs['error'].isna()].copy() if 'error' in rs else rs.copy()
rs['correct'] = rs['correct'].astype(int)

acc2 = egoev.accuracy_table(rs)
acc2['tokens / frame'] = rs.groupby('condition').n_image_tokens.mean() / acc2['frames_per_q']
acc2['saving vs 256'] = 100 * (1 - acc2['tokens / frame'] / 256)
acc2['frame px'] = rs.groupby('condition').frame_px.agg(lambda s_: '/'.join(sorted({str(x) for l in s_ for x in l})))
acc2 = acc2.loc[[c for c in list(SAVE_CONDS) + REF if c in acc2.index]]
print(f'{rs.qid.nunique()} questions, policy D frames (rows after the line of 448-px conditions are section 8\'s 1344 px runs):')
display(acc2[['n', 'accuracy', 'ci_low', 'ci_high', 'tokens / frame', 'saving vs 256', 'frame px']].round(1))

rows = []
for p in picks.values():
    f_, u_ = fov448(p), f"D + uniform, same tokens as fovea@448 ~{p['label']}"
    for a_, b_ in ((f_, u_), (f_, U448)):
        if {a_, b_} <= set(rs.condition):
            rows.append(dict(budget=p['label'], **egoev.paired_test(rs, a_, b_)))
c_ = fov448(p_mid, 'centre')
if {fov448(p_mid), c_} <= set(rs.condition):
    rows.append(dict(budget=p_mid['label'], **egoev.paired_test(rs, fov448(p_mid), c_)))
pt2 = pd.DataFrame(rows)
print('paired on the same questions (diff = a - b, accuracy points; CI from resampling videos):')
display(pt2[['budget', 'a', 'b', 'diff', 'ci_low', 'ci_high', 'a_only_right', 'b_only_right', 'p_mcnemar']].round(3))

fig, ax = plt.subplots(figsize=(7, 4.4))
def curve(names, style, label):
    d = acc2.loc[[n for n in names if n in acc2.index]].sort_values('tokens / frame')
    ax.errorbar(d['tokens / frame'], d.accuracy, yerr=[d.accuracy - d.ci_low, d.ci_high - d.accuracy],
                fmt=style, capsize=3, label=label)
curve([fov448(p) for p in picks.values()], 'o-', 'fovea at gaze (448 px frame)')
curve([f"D + uniform, same tokens as fovea@448 ~{p['label']}" for p in picks.values()], 's--',
      'uniform downscaled to the same tokens')
curve([U448], 'k*', 'uniform 448 (Qwen normal)')
curve([fov448(p_mid, 'centre')], 'x', 'fovea at image centre')
ax.set(xlabel='visual tokens per frame', ylabel='accuracy (%)',
       title=f'policy D frames, {rs.qid.nunique()} questions (95% CI over videos)')
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---

## Reading it

| look at | meaning |
|---|---|
| section 7 parity | must pass, or nothing after it is trustworthy |
| **fovea at gaze vs uniform, same tokens** (paired) | the main result: does putting the tokens at the gaze beat spreading them evenly, at equal cost? |
| fovea at gaze vs fovea at centre | is it the gaze, or just the centre bias of head-mounted video? |
| uniform same tokens vs uniform 448 | sanity: the two uniform conditions have about the same budget and should be close |
| by target in fovea | the gain should come from questions whose target was in the fovea |
| section 10 plot | token saving is real only where the fovea curve sits above the uniform-downscaling curve |
| section 6C | does the gate send when the gaze is a fixation (a reliable fovea)? |

## Caveats

- Training-free: Qwen never saw fractional patch positions or shrunk cells next to
  full-resolution ones. Section 7's drift table shows how far the features move.
- EgoEverything's questions come from gazed-at objects, which favours gaze; the centre
  control and section 6A bound how much.
- Qwen2.5-VL-3B, 12 videos, one with 111 questions: intervals are wide.